# Relace (vztahy) mezi tabulkami — 1:1, 1:N, M:N

Jedním z hlavních cílů kvalitního návrhu databáze je **odstranit redundanci dat** (duplicitní / opakující se data). Jednoduše řečeno — **každá informace existuje v databázi pouze jednou**.

Toho dosáhneme rozdělením dat do více tabulek s různými entitami. Tabulky pak propojíme pomocí **vlastních** (`PRIMARY KEY`) a **cizích** (`FOREIGN KEY`) klíčů. Aby však propojení fungovalo správně, musíme pochopit **typy vztahů** (relací) mezi tabulkami.

---

## Přehled vztahů

| Vztah | Popis | Příklad |
|---|---|---|
| **1:1** | Jednomu záznamu v tabulce A odpovídá **právě jeden** záznam v tabulce B (a naopak) | Občan ↔ Trvalé bydliště |
| **1:N** | Jednomu záznamu v tabulce A odpovídá **více** záznamů v tabulce B | Skladatel → Písně |
| **M:N** | Více záznamům v tabulce A odpovídá **více** záznamů v tabulce B (a naopak) | Zpěváci ↔ Písně |

### Jak se vztahy realizují v SQL?

| Vztah | Realizace v SQL |
|---|---|
| **1:1** | `FOREIGN KEY` + `UNIQUE` na sloupci s cizím klíčem |
| **1:N** | `FOREIGN KEY` (bez `UNIQUE`) |
| **M:N** | Třetí **propojovací (vazební) tabulka** se dvěma cizími klíči |

> **Pozn.:** Rozdíl mezi 1:1 a 1:N v SQL je pouze ve slově `UNIQUE` na sloupci s cizím klíčem.

## Instalace balíčku

In [ ]:
# Instalace knihovny (stačí spustit jednou)
! pip install mysql-connector-python

# Připojení k databázi

Než začneme s databází pracovat, musíme se k ní připojit. Přihlašovací údaje importujeme z modulu `pripojeni.py` (nikdy je nepíšeme přímo do kódu).

In [ ]:
import mysql.connector
from pripojeni import *  # importuje konstanty HOST, USER, PASSWORD, DATABASE

# Připojení k databázi
mydb = mysql.connector.connect(
    host=HOST,
    user=USER,
    password=PASSWORD,
    database=DATABASE
)

# Kurzor — objekt, přes který posíláme SQL příkazy a čteme výsledky
mycursor = mydb.cursor()

---

# Vztah 1:1 (one-to-one)

Jednomu záznamu v první tabulce odpovídá **právě jeden** záznam v druhé tabulce a naopak.

## Příklad z praxe

Tabulka `obcane` a tabulka `trvala_bydliste` — každý občan má právě jedno trvalé bydliště a na jedné adrese má trvalé bydliště právě jeden občan.

<img src="https://raw.githubusercontent.com/JaroslavHolecek/Teaching/master/JupyterNotebook/SQL/Images/JupiterNotebook_11_vztah.png" alt="Příklad vztahu 1:1" width="500">

## Jak se realizuje v SQL?

Cizí klíč (`FOREIGN KEY`) + omezení `UNIQUE` na sloupci s cizím klíčem. Díky `UNIQUE` nemůže mít více občanů stejné trvalé bydliště.

```sql
CREATE TABLE trvala_bydliste (
    id INT PRIMARY KEY AUTO_INCREMENT,
    adresa VARCHAR(100) UNIQUE NOT NULL
);

CREATE TABLE obcane (
    id INT PRIMARY KEY AUTO_INCREMENT,
    jmeno VARCHAR(50),
    prijmeni VARCHAR(50),
    bydliste_id INT UNIQUE,                          -- UNIQUE = max. 1 občan na adresu
    FOREIGN KEY (bydliste_id) REFERENCES trvala_bydliste(id)
);
```

> **Tip:** Pokud je vztah opravdu 1:1 a obě tabulky mají vždy stejný počet záznamů, zvažte, zda nemá smysl tabulky **sloučit do jedné**. Oddělení do dvou tabulek má smysl, pokud např. jedna z tabulek obsahuje citlivé údaje nebo se k ní přistupuje méně často.

### Příklad v Pythonu

In [ ]:
# Úklid z předchozího spuštění
mycursor.execute("DROP TABLE IF EXISTS obcane")
mycursor.execute("DROP TABLE IF EXISTS trvala_bydliste")

# Tabulka trvalých bydlišť
mycursor.execute("""
    CREATE TABLE trvala_bydliste (
        id INT PRIMARY KEY AUTO_INCREMENT,
        adresa VARCHAR(100) UNIQUE NOT NULL
    )
""")

# Tabulka občanů — cizí klíč s UNIQUE = vztah 1:1
mycursor.execute("""
    CREATE TABLE obcane (
        id INT PRIMARY KEY AUTO_INCREMENT,
        jmeno VARCHAR(50) NOT NULL,
        prijmeni VARCHAR(50) NOT NULL,
        bydliste_id INT UNIQUE,
        FOREIGN KEY (bydliste_id) REFERENCES trvala_bydliste(id)
    )
""")

mydb.commit()
print("✅ Tabulky 'trvala_bydliste' a 'obcane' vytvořeny (vztah 1:1).")

Vložíme ukázková data a ověříme, že vztah funguje:

In [ ]:
# Vložíme bydliště
mycursor.execute("""
    INSERT INTO trvala_bydliste (adresa) VALUES
        ('Hlavní 1, Praha'),
        ('Nádražní 5, Kladno'),
        ('Krátká 12, Brno')
""")

# Vložíme občany — každý má unikátní bydliste_id
mycursor.execute("""
    INSERT INTO obcane (jmeno, prijmeni, bydliste_id) VALUES
        ('Jan', 'Novak', 1),
        ('Petra', 'Svobodova', 2),
        ('Karel', 'Dvorak', 3)
""")

mydb.commit()
print("✅ Data vložena.")

# Ověření — pokus o přiřazení stejného bydliště druhému občanovi
try:
    mycursor.execute("""
        INSERT INTO obcane (jmeno, prijmeni, bydliste_id) VALUES
            ('Marie', 'Cerná', 1)
    """)
    mydb.commit()
except mysql.connector.Error as chyba:
    print(f"⛔ Očekávaná chyba — nelze přiřadit stejné bydliště: {chyba}")
    mydb.rollback()

> **Vysvětlení:** Pokus o vložení Marie Černé s `bydliste_id = 1` selže, protože `bydliste_id` má omezení `UNIQUE` a hodnota 1 už je přiřazena Janu Novákovi. Tím je zajištěn vztah 1:1.

---

# Vztah 1:N (one-to-many)

Jednomu záznamu v první tabulce odpovídá **více záznamů** v druhé tabulce, ale každý záznam v druhé tabulce patří **právě jednomu** záznamu v první.

## Příklad z praxe

Tabulka `skladatele` a tabulka `pisne` — skladatel může složit mnoho písní, ale každá píseň má právě jednoho skladatele.

<img src="https://raw.githubusercontent.com/JaroslavHolecek/Teaching/master/JupyterNotebook/SQL/Images/JupiterNotebook_1N_vztah.png" alt="Příklad vztahu 1:N" width="500">

## Jak se realizuje v SQL?

Cizí klíč (`FOREIGN KEY`) **bez** `UNIQUE` — více písní tak může odkazovat na stejného skladatele.

```sql
CREATE TABLE skladatele (
    id INT PRIMARY KEY AUTO_INCREMENT,
    jmeno VARCHAR(50) NOT NULL,
    prijmeni VARCHAR(50) NOT NULL
);

CREATE TABLE pisne (
    id INT PRIMARY KEY AUTO_INCREMENT,
    nazev VARCHAR(100) NOT NULL,
    skladatel_id INT,
    FOREIGN KEY (skladatel_id) REFERENCES skladatele(id)
);
```

> **Důležité:** Rozdíl oproti 1:1 je právě v tom, že sloupec `skladatel_id` **nemá** omezení `UNIQUE`. Díky tomu může více písní odkazovat na stejného skladatele.

### Proč nedat vše do jedné tabulky?

Kdybyste písně zapsali přímo do tabulky skladatelů (např. `pisen_1`, `pisen_2`, …), narazíte na problémy:
- Při každé nové písni byste museli **přidat nový sloupec** — struktura tabulky by se neustále měnila.
- Pokud má skladatel 2 písně a jiný 50, většina sloupců bude **prázdná** (`NULL`).
- **Vyhledávání** a agregace (kolik písní?, nejnovější píseň?) by byly velmi komplikované.

### Příklad v Pythonu

In [ ]:
# Úklid z předchozího spuštění
mycursor.execute("DROP TABLE IF EXISTS pisne")
mycursor.execute("DROP TABLE IF EXISTS skladatele")

# Tabulka skladatelů
mycursor.execute("""
    CREATE TABLE skladatele (
        id INT PRIMARY KEY AUTO_INCREMENT,
        jmeno VARCHAR(50) NOT NULL,
        prijmeni VARCHAR(50) NOT NULL
    )
""")

# Tabulka písní — cizí klíč BEZ UNIQUE = vztah 1:N
mycursor.execute("""
    CREATE TABLE pisne (
        id INT PRIMARY KEY AUTO_INCREMENT,
        nazev VARCHAR(100) NOT NULL,
        skladatel_id INT NOT NULL,
        FOREIGN KEY (skladatel_id) REFERENCES skladatele(id)
    )
""")

mydb.commit()
print("✅ Tabulky 'skladatele' a 'pisne' vytvořeny (vztah 1:N).")

In [ ]:
# Vložíme skladatele
mycursor.execute("""
    INSERT INTO skladatele (jmeno, prijmeni) VALUES
        ('Bedrich', 'Smetana'),
        ('Antonin', 'Dvorak')
""")

# Vložíme písně — více písní může mít stejného skladatele (1:N)
mycursor.execute("""
    INSERT INTO pisne (nazev, skladatel_id) VALUES
        ('Vltava', 1),
        ('Prodana nevesta', 1),
        ('Novosvětská symfonie', 2),
        ('Rusalka', 2),
        ('Slavnostní pochod', 2)
""")

mydb.commit()

# Ověření — výpis všech písní se skladatelem
mycursor.execute("""SELECT skladatele.jmeno, skladatele.prijmeni, pisne.nazev FROM pisne
    JOIN skladatele ON pisne.skladatel_id = skladatele.id""")

for jmeno, prijmeni, nazev in mycursor.fetchall():
    print(f"{jmeno} {prijmeni} — {nazev}")

> **Pozn.:** Příkaz `JOIN` (spojení tabulek) bude podrobněji vysvětlen v pozdějším notebooku. Zde ho používáme pouze pro ukázku, abychom mohli přehledně vypsat data ze dvou propojených tabulek.

---

# Vztah M:N (many-to-many)

Více záznamům v první tabulce odpovídá **více záznamů** v druhé tabulce a naopak.

## Příklad z praxe

Tabulka `zpevaci` a tabulka `pisne` — zpěvák může zpívat více písní a jednu píseň může zpívat více zpěváků.

<img src="https://raw.githubusercontent.com/JaroslavHolecek/Teaching/master/JupyterNotebook/SQL/Images/JupiterNotebook_MN_vztah.png" alt="Příklad vztahu M:N" width="500">

## Jak se realizuje v SQL?

Vztah M:N **nelze realizovat přímo** dvěma tabulkami. Potřebujeme třetí **propojovací (vazební) tabulku**, která obsahuje:
- Cizí klíč na první tabulku
- Cizí klíč na druhou tabulku
- Volitelně další atributy (např. datum, poznámka)

```sql
CREATE TABLE zpevaci (
    id INT PRIMARY KEY AUTO_INCREMENT,
    jmeno VARCHAR(50) NOT NULL,
    prijmeni VARCHAR(50) NOT NULL
);

CREATE TABLE pisne (
    id INT PRIMARY KEY AUTO_INCREMENT,
    nazev VARCHAR(100) NOT NULL
);

-- Propojovací tabulka
CREATE TABLE interpretace (
    id INT PRIMARY KEY AUTO_INCREMENT,
    zpevak_id INT NOT NULL,
    pisen_id INT NOT NULL,
    datum_nahravky DATE,
    FOREIGN KEY (zpevak_id) REFERENCES zpevaci(id),
    FOREIGN KEY (pisen_id) REFERENCES pisne(id),
    CONSTRAINT unikatni_interpretace UNIQUE (zpevak_id, pisen_id)
);
```

> **Konvence:** Propojovací tabulku pojmenováváme buď kombinací názvů obou tabulek (např. `zpevaci_pisne`), nebo výstižným názvem vztahu (např. `interpretace`, `objednavky_produkty`).

> **Tip:** Omezení `UNIQUE (zpevak_id, pisen_id)` zabrání duplicitnímu záznamu — jeden zpěvák nemůže mít stejnou píseň přiřazenou dvakrát.

### Příklad v Pythonu

In [ ]:
# Úklid z předchozího spuštění
mycursor.execute("DROP TABLE IF EXISTS interpretace")
mycursor.execute("DROP TABLE IF EXISTS zpevaci")
mycursor.execute("DROP TABLE IF EXISTS pisne")

# Tabulka zpěváků
mycursor.execute("""
    CREATE TABLE zpevaci (
        id INT PRIMARY KEY AUTO_INCREMENT,
        jmeno VARCHAR(50) NOT NULL,
        prijmeni VARCHAR(50) NOT NULL
    )
""")

# Tabulka písní
mycursor.execute("""
    CREATE TABLE pisne (
        id INT PRIMARY KEY AUTO_INCREMENT,
        nazev VARCHAR(100) NOT NULL
    )
""")

# Propojovací tabulka — realizuje vztah M:N
mycursor.execute("""
    CREATE TABLE interpretace (
        id INT PRIMARY KEY AUTO_INCREMENT,
        zpevak_id INT NOT NULL,
        pisen_id INT NOT NULL,
        datum_nahravky DATE,
        FOREIGN KEY (zpevak_id) REFERENCES zpevaci(id),
        FOREIGN KEY (pisen_id) REFERENCES pisne(id),
        CONSTRAINT unikatni_interpretace UNIQUE (zpevak_id, pisen_id)
    )
""")

mydb.commit()
print("✅ Tabulky 'zpevaci', 'pisne' a 'interpretace' vytvořeny (vztah M:N).")

In [ ]:
# Vložíme zpěváky
mycursor.execute("""
    INSERT INTO zpevaci (jmeno, prijmeni) VALUES
        ('Karel', 'Gott'),
        ('Lucie', 'Bila'),
        ('Jaromir', 'Nohavica')
""")

# Vložíme písně
mycursor.execute("""
    INSERT INTO pisne (nazev) VALUES
        ('Lady Karneval'),
        ('Být stále mlád'),
        ('Tři čuníci')
""")

# Propojíme — kdo co nazpíval
mycursor.execute("""
    INSERT INTO interpretace (zpevak_id, pisen_id, datum_nahravky) VALUES
        (1, 1, '1977-01-01'),
        (1, 2, '1997-06-15'),
        (2, 2, '2005-03-20'),
        (3, 3, '1995-11-10'),
        (2, 3, '2010-08-01')
""")

mydb.commit()

# Výpis — kdo co nazpíval
mycursor.execute("""
    SELECT zpevaci.jmeno, zpevaci.prijmeni, pisne.nazev, interpretace.datum_nahravky
    FROM interpretace
    JOIN zpevaci ON interpretace.zpevak_id = zpevaci.id
    JOIN pisne ON interpretace.pisen_id = pisne.id
""")

for jmeno, prijmeni, nazev, datum in mycursor.fetchall():
    print(f"{jmeno} {prijmeni} — {nazev} ({datum})")

---

# Shrnutí — porovnání vztahů

| Vlastnost | 1:1 | 1:N | M:N |
|---|---|---|---|
| Počet tabulek | 2 | 2 | 3 (+ propojovací) |
| Cizí klíč | ✅ + `UNIQUE` | ✅ (bez `UNIQUE`) | ✅ ✅ (dva v propojovací tabulce) |
| Příklad | Občan ↔ Bydliště | Skladatel → Písně | Zpěváci ↔ Písně |

### Schéma příkladu z praxe

Následující obrázek znázorňuje, jak může vypadat databáze s použitím vztahů 1:N a M:N:

<img src="https://raw.githubusercontent.com/JaroslavHolecek/Teaching/master/JupyterNotebook/SQL/Images/JupiterNotebook_PrikladDatabaze.png" alt="Příklad vztahů v databázi" width="600">

> V tabulkách jsou vidět i „propojovací tabulky

---

# Cvičení

Zde bude následovat série úkolů, díky kterým si ověříte, zda jste látku pochopili.

> V každém cvičení se nejprve připojte k databázi pomocí konstant z modulu `pripojeni`. Na konci se vždy odpojte.

## Cvičení 1 — Určení typu vztahu (teorie)

U následujících dvojic tabulek určete, o jaký typ vztahu se jedná a ve které tabulce bude cizí klíč.

### a) Mazlíčci a majitelé

| `mazlicci` | `majitele` |
|---|---|
| `id` INT PK | `id` INT PK |
| `druh` VARCHAR(30) | `jmeno` VARCHAR(50) |
| `jmeno` VARCHAR(30) | `prijmeni` VARCHAR(50) |
| | `mazlicek_id` INT FK |

- **Vztah:** <!-- TODO: doplňte 1:1 / 1:N / M:N -->
- **Cizí klíč v tabulce:** <!-- TODO: doplňte -->
- **Zdůvodnění:** <!-- TODO: doplňte -->

---

### b) Pracovníci a služební auta

| `pracovnici` | `sluzebni_auta` |
|---|---|
| `id` INT PK | `id` INT PK |
| `jmeno` VARCHAR(50) | `znacka` VARCHAR(30) |
| `prijmeni` VARCHAR(50) | `spz` CHAR(7) |
| | `ridic_id` INT FK UNIQUE |

- **Vztah:** <!-- TODO: doplňte 1:1 / 1:N / M:N -->
- **Cizí klíč v tabulce:** <!-- TODO: doplňte -->
- **Zdůvodnění:** <!-- TODO: doplňte -->

---

### c) Učitelé a předměty

| `ucitele` | `predmety` |
|---|---|
| `id` INT PK | `id` INT PK |
| `jmeno` VARCHAR(50) | `nazev` VARCHAR(50) |
| `prijmeni` VARCHAR(50) | |
| `katedra` VARCHAR(30) | |

Jeden učitel může učit více předmětů a jeden předmět může učit více učitelů.

- **Vztah:** <!-- TODO: doplňte 1:1 / 1:N / M:N -->
- **Potřebujeme propojovací tabulku?** <!-- TODO: ano/ne -->
- **Jak by se jmenovala?** <!-- TODO: doplňte -->

## Cvičení 2 — Vztah 1:N v praxi

Vytvořte dvě tabulky:

### Tabulka `tridy`
- `id` — INT, PRIMARY KEY, AUTO_INCREMENT
- `nazev` — CHAR(10), NOT NULL, UNIQUE (např. `'EP4B'`)
- `ucebna` — CHAR(10)

### Tabulka `studenti`
- `id` — INT, PRIMARY KEY, AUTO_INCREMENT
- `jmeno` — VARCHAR(50), NOT NULL
- `prijmeni` — VARCHAR(50), NOT NULL
- `trida_id` — INT, FOREIGN KEY → `tridy(id)`

Potom vložte alespoň **2 třídy** a **4 studenty** (alespoň 2 studenti ve stejné třídě).

<details>
<summary>🔎 Očekávaný výstup (klikni pro zobrazení)</summary>

```
Tabulka tridy vytvořena správně.
Tabulka studenti vytvořena správně.
✅ Data vložena — X studentů v Y třídách.
```

</details>

In [ ]:
import mysql.connector
from pripojeni import *

mydb = mysql.connector.connect(
    host=HOST, user=USER, password=PASSWORD, database=DATABASE
)
mycursor = mydb.cursor()

# Úklid
mycursor.execute("DROP TABLE IF EXISTS studenti")
mycursor.execute("DROP TABLE IF EXISTS tridy")
mydb.commit()

# TODO: Vytvořte tabulky 'tridy' a 'studenti' (vztah 1:N) →


# TODO: Vložte alespoň 2 třídy a 4 studenty →


mydb.commit()

# --- tuto část neměnit! ---
mycursor.execute("DESCRIBE tridy")
table = str(mycursor.fetchall()).replace("int(11)", "int")
expected = "[('id', 'int', 'NO', 'PRI', None, 'auto_increment'), ('nazev', 'char(10)', 'NO', 'UNI', None, ''), ('ucebna', 'char(10)', 'YES', '', None, '')]"
if table == expected:
    print("Tabulka tridy vytvořena správně.")
else:
    print("Tabulka tridy není dle zadání.\nVaše tabulka:")
    print(table)
    print("Očekávaná tabulka:")
    print(expected)

mycursor.execute("DESCRIBE studenti")
table = str(mycursor.fetchall()).replace("int(11)", "int")
expected = "[('id', 'int', 'NO', 'PRI', None, 'auto_increment'), ('jmeno', 'varchar(50)', 'NO', '', None, ''), ('prijmeni', 'varchar(50)', 'NO', '', None, ''), ('trida_id', 'int', 'YES', 'MUL', None, '')]"
if table == expected:
    print("Tabulka studenti vytvořena správně.")
else:
    print("Tabulka studenti není dle zadání.\nVaše tabulka:")
    print(table)
    print("Očekávaná tabulka:")
    print(expected)

mycursor.execute("SELECT COUNT(*) FROM studenti")
pocet_studentu = mycursor.fetchone()[0]
mycursor.execute("SELECT COUNT(*) FROM tridy")
pocet_trid = mycursor.fetchone()[0]
if pocet_studentu >= 4 and pocet_trid >= 2:
    print(f"✅ Data vložena — {pocet_studentu} studentů v {pocet_trid} třídách.")
else:
    print(f"❌ Málo dat: {pocet_studentu} studentů (min. 4), {pocet_trid} tříd (min. 2).")

# Úklid
mycursor.execute("DROP TABLE IF EXISTS studenti")
mycursor.execute("DROP TABLE IF EXISTS tridy")
mydb.commit()
mycursor.close()
mydb.close()

## Cvičení 3 — Vztah M:N v praxi

Vytvořte tři tabulky pro evidenci, kteří studenti jsou zapsáni do kterých kroužků:

### Tabulka `studenti`
- `id` — INT, PRIMARY KEY, AUTO_INCREMENT
- `jmeno` — VARCHAR(50), NOT NULL
- `prijmeni` — VARCHAR(50), NOT NULL

### Tabulka `krouzky`
- `id` — INT, PRIMARY KEY, AUTO_INCREMENT
- `nazev` — VARCHAR(50), NOT NULL, UNIQUE
- `den` — CHAR(10) (den v týdnu)

### Propojovací tabulka `zapisy`
- `id` — INT, PRIMARY KEY, AUTO_INCREMENT
- `student_id` — INT, NOT NULL, FOREIGN KEY → `studenti(id)`
- `krouzek_id` — INT, NOT NULL, FOREIGN KEY → `krouzky(id)`
- `CONSTRAINT unikatni_zapis UNIQUE (student_id, krouzek_id)`

Vložte alespoň **3 studenty**, **2 kroužky** a **4 zápisy** (alespoň 1 student ve 2 kroužcích).

<details>
<summary>🔎 Očekávaný výstup (klikni pro zobrazení)</summary>

```
✅ Tabulky a data vytvořeny správně.
Jan Novak — Robotika
Jan Novak — Programování
...
```

</details>

In [ ]:
import mysql.connector
from pripojeni import *

mydb = mysql.connector.connect(
    host=HOST, user=USER, password=PASSWORD, database=DATABASE
)
mycursor = mydb.cursor()

# Úklid
mycursor.execute("DROP TABLE IF EXISTS zapisy")
mycursor.execute("DROP TABLE IF EXISTS studenti")
mycursor.execute("DROP TABLE IF EXISTS krouzky")
mydb.commit()

# TODO: Vytvořte tabulky 'studenti', 'krouzky' a 'zapisy' (vztah M:N) →


# TODO: Vložte data →


mydb.commit()

# --- tuto část neměnit! ---
mycursor.execute("SELECT COUNT(*) FROM zapisy")
pocet_zapisu = mycursor.fetchone()[0]
if pocet_zapisu >= 4:
    print("✅ Tabulky a data vytvořeny správně.")
else:
    print(f"❌ V tabulce zapisy je pouze {pocet_zapisu} záznamů (minimum je 4).")

# Výpis zápisů
mycursor.execute("""
    SELECT studenti.jmeno, studenti.prijmeni, krouzky.nazev
    FROM zapisy
    JOIN studenti ON zapisy.student_id = studenti.id
    JOIN krouzky ON zapisy.krouzek_id = krouzky.id
""")
for jmeno, prijmeni, krouzek in mycursor.fetchall():
    print(f"{jmeno} {prijmeni} — {krouzek}")

# Úklid
mycursor.execute("DROP TABLE IF EXISTS zapisy")
mycursor.execute("DROP TABLE IF EXISTS studenti")
mycursor.execute("DROP TABLE IF EXISTS krouzky")
mydb.commit()
mycursor.close()
mydb.close()

## Cvičení 4 — Kompletní úloha

Celý úkol je na vás od začátku do konce. Navrhněte a vytvořte databázi pro **e-shop** s těmito tabulkami:

### Tabulka `zakaznici`
- `id` — INT, PRIMARY KEY, AUTO_INCREMENT
- `jmeno` — VARCHAR(50), NOT NULL
- `prijmeni` — VARCHAR(50), NOT NULL
- `email` — VARCHAR(100), NOT NULL, UNIQUE

### Tabulka `produkty`
- `id` — INT, PRIMARY KEY, AUTO_INCREMENT
- `nazev` — VARCHAR(100), NOT NULL
- `cena` — DECIMAL(10, 2), NOT NULL

### Tabulka `objednavky`
- `id` — INT, PRIMARY KEY, AUTO_INCREMENT
- `zakaznik_id` — INT, NOT NULL, FOREIGN KEY → `zakaznici(id)` *(vztah 1:N)*
- `datum` — DATE, NOT NULL

### Propojovací tabulka `polozky_objednavek`
- `id` — INT, PRIMARY KEY, AUTO_INCREMENT
- `objednavka_id` — INT, NOT NULL, FOREIGN KEY → `objednavky(id)` *(vztah M:N)*
- `produkt_id` — INT, NOT NULL, FOREIGN KEY → `produkty(id)` *(vztah M:N)*
- `pocet_kusu` — INT, NOT NULL, DEFAULT 1

**Úkoly:**
1. Vytvořte všechny tabulky.
2. Vložte alespoň **2 zákazníky**, **3 produkty**, **2 objednávky** a **4 položky objednávek**.
3. Vypište, kdo si co objednal (pomocí `JOIN`).
4. Smažte tabulky (ve správném pořadí!) a odpojte se.

> **Pozor na pořadí mazání:** Nejprve mažte tabulky s cizími klíči, pak teprve ty, na které se odkazují.

<details>
<summary>🔎 Očekávaný výstup (klikni pro zobrazení)</summary>

```
--- Objednávky ---
Jan Novak — Notebook (2 ks) — 2025-01-15
Jan Novak — Myš (1 ks) — 2025-01-15
Petra Svobodova — Klávesnice (3 ks) — 2025-02-20
...
```

</details>

In [ ]:
import mysql.connector
from pripojeni import *

mydb = mysql.connector.connect(
    host=HOST, user=USER, password=PASSWORD, database=DATABASE
)
mycursor = mydb.cursor()

# TODO: Celý úkol je na vás →


# Nezapomeňte na konci smazat tabulky (ve správném pořadí) a odpojit se!